# Synthetic elections data
Each column is sampled independently from the category proportions observed in `private_data_preprocessed.csv` (196 rows).
Relationships between columns are **not** preserved, and no real record is copied.

In [27]:
import numpy as np
import pandas as pd

## Settings and distributions

In [28]:
N_ROWS = 1000
SEED = 42

# column -> {value: probability}
DISTRIBUTIONS = {
    "sex":            {"Female": 0.505, "Male": 0.495},
    "evote":          {"0": 0.69, "1": 0.31},
    "zip":            {"2400": 0.37, "2200": 0.24, "2100": 0.20, "2300": 0.19},
    "education":      {"Medium": 0.43, "High": 0.37, "Low": 0.20},
    "citizenship":    {"Danish": 0.88, "non-danish": 0.12},
    "marital_status": {"Married/separated": 0.44, "Never married": 0.36,
                       "Divorced": 0.15, "Widowed": 0.05},
    "party":          {"Green": 0.63, "Red": 0.34, "Invalid vote": 0.03},
    "age_group":      {"45-64": 0.41, "30-44": 0.25, "65 or older": 0.21, "18-29": 0.13},
}

## Generator

In [29]:
def generate(n_rows=N_ROWS, seed=SEED):
    rng = np.random.default_rng(seed)
    data = {"name": ["*"] * n_rows}  # name is fully suppressed in the original
    for col, dist in DISTRIBUTIONS.items():
        values, probs = list(dist.keys()), np.array(list(dist.values()))
        data[col] = rng.choice(values, size=n_rows, p=probs / probs.sum())
    return pd.DataFrame(data, dtype=str)

synth = generate()
synth

,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Male,0,2300,Low,Danish,Married/separated,Green,65 or older
1,*,Female,0,2200,High,Danish,Never married,Green,18-29
2,*,Male,0,2300,Medium,Danish,Never married,Green,30-44
3,*,Male,0,2100,High,Danish,Married/separated,Green,30-44
4,*,Female,0,2400,Medium,Danish,Married/separated,Green,18-29
...,...,...,...,...,...,...,...,...,...
995,*,Female,1,2200,Medium,Danish,Married/separated,Red,45-64
996,*,Male,0,2100,Medium,Danish,Never married,Red,45-64
997,*,Female,0,2400,High,Danish,Divorced,Green,30-44
998,*,Male,1,2300,Medium,Danish,Divorced,Red,45-64


## Sanity check: category proportions

In [30]:
for col in DISTRIBUTIONS:
    print(synth[col].value_counts(normalize=True).round(2).to_dict())

{'Female': 0.51, 'Male': 0.49}
{'0': 0.68, '1': 0.32}
{'2400': 0.37, '2200': 0.26, '2300': 0.19, '2100': 0.18}
{'Medium': 0.44, 'High': 0.37, 'Low': 0.2}
{'Danish': 0.9, 'non-danish': 0.1}
{'Married/separated': 0.46, 'Never married': 0.36, 'Divorced': 0.14, 'Widowed': 0.04}
{'Green': 0.64, 'Red': 0.33, 'Invalid vote': 0.03}
{'45-64': 0.42, '30-44': 0.24, '65 or older': 0.2, '18-29': 0.14}


## Add our "real" dataset

In [31]:
real = pd.read_csv(r"C:\Users\filip\Dokumenty\Data Science\Data Science 5th Semester\sec priv\Elections-Data-Anonymisation-Security-Privacy-Project\data\private_data_preprocessed.csv", dtype=str)
real = real.drop(columns=[c for c in real.columns if c.startswith("Unnamed") or c == "X"], errors="ignore")  # drop row-index column
real


,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Female,0,2100,Medium,Danish,Never married,Green,18-29
1,*,Female,0,2300,Medium,Danish,Never married,Red,45-64
2,*,Female,1,2400,High,Danish,Never married,Green,45-64
3,*,Male,1,2400,High,Danish,Never married,Red,65 or older
4,*,Male,0,2100,Medium,Danish,Never married,Green,30-44
...,...,...,...,...,...,...,...,...,...
191,*,Male,0,2400,Medium,Danish,Married/separated,Green,65 or older
192,*,Male,1,2400,Medium,Danish,Married/separated,Green,45-64
193,*,Female,0,2400,Medium,Danish,Married/separated,Green,45-64
194,*,Female,1,2300,High,Danish,Married/separated,Invalid vote,30-44


## Combine and shuffle

In [36]:
combined = pd.concat([real, synth], ignore_index=True)
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
combined
df = combined
df["zip"] = df["zip"].str[0] + "***"
df

,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Male,0,2***,Medium,Danish,Married/separated,Green,18-29
1,*,Female,1,2***,High,Danish,Divorced,Green,45-64
2,*,Male,0,2***,Medium,Danish,Married/separated,Green,30-44
3,*,Female,1,2***,High,Danish,Never married,Red,45-64
4,*,Female,1,2***,High,Danish,Never married,Green,18-29
...,...,...,...,...,...,...,...,...,...
1191,*,Female,0,2***,Medium,Danish,Married/separated,Green,45-64
1192,*,Male,0,2***,Medium,Danish,Married/separated,Red,45-64
1193,*,Male,1,2***,High,Danish,Married/separated,Red,30-44
1194,*,Female,1,2***,Medium,Danish,Married/separated,Green,45-64


## Quasi and sensitive variables

In [37]:
qi   = ["sex", "evote", "zip", "education", "citizenship", "age_group", "marital_status"]
sens = ["party"]

## k-anonymity check
Size of each equivalence class. `dropna=False` keeps suppressed (`NaN`) values as their own category, like sdcMicro does.

In [41]:
def k_groups(data, qi):
    return data.groupby(qi, dropna=False).size().reset_index(name="n")

K_TARGET = 3

groups = k_groups(df, qi)
print("current k:", groups["n"].min())
print(f"groups violating k = {K_TARGET}:", (groups["n"] < K_TARGET).sum())
print(f"records violating k = {K_TARGET}:", groups.loc[groups["n"] < K_TARGET, "n"].sum())


current k: 1
groups violating k = 3: 111
records violating k = 3: 159


## Local suppression to reach k
sdcMicro's `kAnon()` has no Python equivalent, so this is a simple greedy version: 
for records in classes smaller than k, set one quasi-identifier to `NaN`, starting with the least important column, and repeat until every class has at least k records.

`importance` lists the columns in the order they get suppressed (first = suppressed first). I put the columns with the most categories first, since they cause the most violations. Change it to whatever you can best afford to lose.

In [42]:
def k_anon_suppress(data, qi, k, importance=None):
    out = data.copy()
    order = importance or sorted(qi, key=lambda c: -out[c].nunique())   # most categories first
    for col in order:
        size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
        viol = size < k
        if not viol.any():
            break
        out.loc[viol, col] = pd.NA
    # records still in a class < k: suppress all their quasi-identifiers so they merge into one class
    size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
    out.loc[size < k, qi] = pd.NA
    size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
    if (size < k).any():
        print(f"warning: {(size < k).sum()} fully suppressed records still form a class < k={k}; drop them or lower k")
    return out

df_anon = k_anon_suppress(df, qi, k=K_TARGET)

print("k after suppression:", k_groups(df_anon, qi)["n"].min())
print("NaNs introduced per column:")
print(df_anon[qi].isna().sum())
df_anon

k after suppression: 2
NaNs introduced per column:
sex                 2
evote               2
zip                 2
education          18
citizenship         2
age_group         159
marital_status     56
dtype: int64


,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Male,0,2***,Medium,Danish,Married/separated,Green,18-29
1,*,Female,1,2***,High,Danish,Divorced,Green,45-64
2,*,Male,0,2***,Medium,Danish,Married/separated,Green,30-44
3,*,Female,1,2***,High,Danish,Never married,Red,45-64
4,*,Female,1,2***,High,Danish,Never married,Green,18-29
...,...,...,...,...,...,...,...,...,...
1191,*,Female,0,2***,Medium,Danish,Married/separated,Green,45-64
1192,*,Male,0,2***,Medium,Danish,Married/separated,Red,45-64
1193,*,Male,1,2***,High,Danish,Married/separated,Red,30-44
1194,*,Female,1,2***,Medium,Danish,Married/separated,Green,45-64


## fully supressed rows

In [44]:
fully_suppressed = df_anon[df_anon[qi].isna().all(axis=1)]
print(f"{len(fully_suppressed)} fully suppressed rows")
df.loc[fully_suppressed.index]

2 fully suppressed rows


,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
348,*,Male,1,2***,High,non-danish,Married/separated,Red,45-64
1092,*,Male,1,2***,High,non-danish,Divorced,Red,45-64


## l-diversity check
Per equivalence class: size and l for the sensitive attribute `party` (number of distinct parties in the class). `marital_status` is a quasi-identifier, so it is part of the class definition and is not checked here.

In [45]:
def l_diversity(data, qi, sens):
    s = sens[0]                                   # only one sensitive attribute: party
    g = data.groupby(qi, dropna=False)
    res = g.agg(n=(s, "size")).reset_index()
    res["l_party"] = g[s].nunique(dropna=False).values
    return res

def l_report(data, qi, sens, l=2):
    lg = l_diversity(data, qi, sens)
    viol = lg["l_party"] < l
    print("current l (party):          ", lg["l_party"].min())
    print("groups violating l (party): ", viol.sum(), "of", len(lg))
    print("records violating l (party):", lg.loc[viol, "n"].sum(), "of", lg["n"].sum())
    return lg

L_TARGET = 2

### On the original data (`df`)

In [46]:
l_groups = l_report(df, qi, sens, L_TARGET)

# the offending groups, smallest first
l_groups[l_groups["l_party"] < L_TARGET].sort_values(["l_party", "n"]) 

current l (party):           1
groups violating l (party):  98 of 230
records violating l (party): 146 of 1196


,sex,evote,zip,education,citizenship,age_group,marital_status,n,l_party
10,Female,0,2***,High,Danish,45-64,Widowed,1,1
15,Female,0,2***,High,non-danish,18-29,Divorced,1,1
16,Female,0,2***,High,non-danish,18-29,Married/separated,1,1
17,Female,0,2***,High,non-danish,18-29,Never married,1,1
18,Female,0,2***,High,non-danish,18-29,Widowed,1,1
...,...,...,...,...,...,...,...,...,...
179,Male,0,2***,Medium,non-danish,45-64,Never married,3,1
184,Male,1,2***,High,Danish,30-44,Divorced,3,1
135,Male,0,2***,High,Danish,65 or older,Widowed,4,1
161,Male,0,2***,Medium,Danish,18-29,Divorced,4,1


### On the k-anonymized data (`df_anon`)

In [47]:
l_groups_anon = l_report(df_anon, qi, sens, L_TARGET)

l_groups_anon[l_groups_anon["l_party"] < L_TARGET].sort_values(["l_party", "n"])

current l (party):           1
groups violating l (party):  22 of 160
records violating l (party): 77 of 1196


,sex,evote,zip,education,citizenship,age_group,marital_status,n,l_party
159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,1
12,Female,0,2***,High,Danish,NaN,Widowed,3,1
19,Female,0,2***,Low,Danish,30-44,Never married,3,1
20,Female,0,2***,Low,Danish,45-64,Divorced,3,1
29,Female,0,2***,Low,non-danish,NaN,NaN,3,1
66,Female,1,2***,Low,Danish,NaN,Divorced,3,1
68,Female,1,2***,Low,Danish,NaN,NaN,3,1
107,Male,0,2***,Low,Danish,65 or older,Never married,3,1
108,Male,0,2***,Low,Danish,NaN,Divorced,3,1
109,Male,0,2***,Low,Danish,NaN,Widowed,3,1


### Before vs after k-anonymization
Records in classes that fail l-diversity on `party`.

In [53]:
def fail_records(lg, l=L_TARGET):
    return int(lg.loc[lg["l_party"] < l, "n"].sum())

pd.DataFrame({
    "records_total":      [l_groups["n"].sum(), l_groups_anon["n"].sum()],
    "records_failing_l":  [fail_records(l_groups), fail_records(l_groups_anon)],
}, index=["original (df)", "k-anonymized (df_anon)"])

,records_total,records_failing_l
original (df),1196,146
k-anonymized (df_anon),1196,77


## fulfill L=2

In [55]:
def add_opposite_records(data, qi, sens, l=2):
    s = sens[0]
    opposite = {"Green": "Red", "Red": "Green", "Invalid vote": "Green"}
    out = data.copy()
    out["is_added"] = False
    new_rows = []
    for _, grp in data.groupby(qi, dropna=False):
        if grp[s].nunique(dropna=False) < l:
            row = grp.iloc[0].copy()                      # same QI values (incl. suppressed NaNs)
            row[s] = opposite.get(row[s], "Green")        # flip the party
            row["is_added"] = True
            new_rows.append(row)
    added = pd.DataFrame(new_rows)
    return pd.concat([out, added], ignore_index=True)

df_l = add_opposite_records(df_anon, qi, sens, L_TARGET)

print("added records:", df_l["is_added"].sum())
l_groups_final = l_report(df_l, qi, sens, L_TARGET)
l_groups_final[l_groups_final["l_party"] < L_TARGET]     # should be empty


df_l

added records: 22
current l (party):           2
groups violating l (party):  0 of 160
records violating l (party): 0 of 1218


,name,sex,evote,zip,education,citizenship,marital_status,party,age_group,is_added
0,*,Male,0,2***,Medium,Danish,Married/separated,Green,18-29,False
1,*,Female,1,2***,High,Danish,Divorced,Green,45-64,False
2,*,Male,0,2***,Medium,Danish,Married/separated,Green,30-44,False
3,*,Female,1,2***,High,Danish,Never married,Red,45-64,False
4,*,Female,1,2***,High,Danish,Never married,Green,18-29,False
...,...,...,...,...,...,...,...,...,...,...
1213,*,Male,0,2***,Medium,non-danish,Never married,Red,45-64,True
1214,*,Male,0,2***,Medium,non-danish,<NA>,Red,<NA>,True
1215,*,Male,1,2***,High,Danish,Divorced,Green,30-44,True
1216,*,Male,1,2***,Low,Danish,Never married,Red,45-64,True


## create csv with new file - will check significannce

In [59]:
final_df = df_l.drop(columns=['is_added'])
final_df.to_csv("private_data_annonimised.csv")